# Evaluation Measures

**DCG@10 / NDCG@10** — ranking quality (graded metric), range $[0,1]$, higher better:

$$\text{DCG@}10=\sum_{i=1}^{10}\frac{rel_i}{\log_2(i+1)},\qquad
\text{NDCG@}10=\frac{\text{DCG@}10}{\text{IDCG@}10}$$

**Cosine** — dense semantic match (unit vectors), higher better:

$$\operatorname{sim}(q,c)=\langle\hat q,\hat c\rangle$$

**BM25 / IDF** — lexical match, higher better ($k_1{=}1.5,\ b{=}0.75$):

$$\text{BM25}(q,c)=\sum_{\tau\in q}\text{IDF}(\tau)\,\frac{f(\tau,c)(k_1{+}1)}{f(\tau,c)+k_1(1-b+b\,|c|/\overline{|c|})},\qquad
\text{IDF}(\tau)=\ln\!\frac{N-df(\tau)+0.5}{df(\tau)+0.5}$$

**Recall@M** — relevant pages reaching the top-$M$ pool, higher better:

$$\text{Recall@}M=\frac{|\text{gold in top-}M|}{|\text{all gold}|}$$

In [2]:
# ============================================================================
# CHECK CELL — load the current pipeline files and evaluate run() on the public
# queries. Edit CONTROLS, run, read the table. (Reloads modules each run, so it
# always reflects the latest retrieve.py / index.py on disk.)
# ============================================================================
import importlib, time
import numpy as np, pandas as pd

# ------------------------------- CONTROLS -----------------------------------
SHOW_ZERO_IDS   = True     # list the query_ids that scored 0
SHOW_SPLIT      = True     # add single/|rel|=1 vs multi/|rel|>1 rows
OVERRIDE = {               # temporarily override retrieve.py constants (None = use file)
    "W_DENSE": None, "W_BM25": None, "W_CHUNK": None,
    "CAND_M":  None, "CE_TOPK": None, "CHUNK_TOPN": None,
}
# ----------------------------------------------------------------------------

import eval as ev, utils
for _m in ("chunk", "index", "retrieve", "main"):
    importlib.reload(importlib.import_module(_m))
import retrieve, main

for k, v in OVERRIDE.items():                 # apply only the overrides you set
    if v is not None:
        setattr(retrieve, k, v)
retrieve._H = retrieve._CHUNK = None          # drop caches so overrides take effect
retrieve._PID2ROW = None

rows    = ev.load_query_file(utils.PUBLIC_QUERIES_PATH)
queries = [r["query"] for r in rows]
truth   = [set(r["relevant_page_ids"]) for r in rows]
K = ev.K_EVAL

t0 = time.perf_counter()
ranked = main.run(queries)                    # the graded entry point
elapsed = time.perf_counter() - t0

def _dedup(lst):
    seen, out = set(), []
    for x in lst:
        if x not in seen:
            seen.add(x); out.append(x)
    return out

ndcg, rec10, recpool = [], [], []
for r, g in zip(ranked, truth):
    r = _dedup(list(r)); ng = len(g)
    ndcg.append(ev.ndcg_at_k(r, g, k=K))
    rec10.append(len(set(r[:K]) & g) / ng if ng else 0.0)
    recpool.append(len(set(r) & g) / ng if ng else 0.0)
ndcg, rec10, recpool = map(np.array, (ndcg, rec10, recpool))
is_multi = np.array([len(g) > 1 for g in truth])

def _row(mask):
    return {"n": int(mask.sum()),
            "NDCG@10":  round(float(ndcg[mask].mean()), 4),
            "Recall@10":round(float(rec10[mask].mean()), 4),
            "PoolRec":  round(float(recpool[mask].mean()), 4),
            "zeros":    int((ndcg[mask] == 0).sum()),
            "miss":     int((recpool[mask] == 0).sum()),               # gold never retrieved
            "rank>10":  int(((recpool[mask] > 0) & (rec10[mask] == 0)).sum())}

table = {"overall": _row(np.ones(len(ndcg), bool))}
if SHOW_SPLIT:
    table["single |rel|=1"] = _row(~is_multi)
    table["multi  |rel|>1"] = _row(is_multi)
df = pd.DataFrame(table).T[["n","NDCG@10","Recall@10","PoolRec","zeros","miss","rank>10"]]

cfg = {k: getattr(retrieve, k) for k in
       ("W_DENSE","W_BM25","W_CHUNK","CAND_M","CHUNK_TOPN","CE_TOPK")}
print("config:", "  ".join(f"{k}={v}" for k, v in cfg.items()))
print(f"run(): {elapsed:.1f}s / {len(queries)} q  ({elapsed/len(queries)*1000:.0f} ms/q)  "
      f"60s-cap(local): {'OK' if elapsed < 60 else 'OVER'}\n")
print(df.to_string())

if SHOW_ZERO_IDS:
    zids = [rows[i]["query_id"] for i in range(len(ndcg)) if ndcg[i] == 0]
    print(f"\nzero-score queries ({len(zids)}): {zids}")

C:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


config: W_DENSE=0.3  W_BM25=0.5  W_CHUNK=0.2  CAND_M=50  CHUNK_TOPN=256  CE_TOPK=10
run(): 44.9s / 50 q  (897 ms/q)  60s-cap(local): OK

                   n  NDCG@10  Recall@10  PoolRec  zeros  miss  rank>10
overall         50.0   0.3170     0.4367     0.87   20.0   2.0     18.0
single |rel|=1  25.0   0.4587     0.6000     0.92   10.0   2.0      8.0
multi  |rel|>1  25.0   0.1753     0.2733     0.82   10.0   0.0     10.0

zero-score queries (20): ['q_public_006', 'q_public_007', 'q_public_008', 'q_public_013', 'q_public_014', 'q_public_017', 'q_public_019', 'q_public_020', 'q_public_023', 'q_public_024', 'q_public_026', 'q_public_031', 'q_public_037', 'q_public_041', 'q_public_042', 'q_public_043', 'q_public_044', 'q_public_048', 'q_public_049', 'q_public_050']
